In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_project.silver;

In [0]:
# Step 1: Read from Bronze
org_df = spark.table("medical_project.bronze.organizations")

In [0]:
# Step 2: Standardize Column Names
import re

def clean_column(col_name):
    col_name = col_name.strip().lower()
    col_name = re.sub(r"[^\w]", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name)
    col_name = col_name.strip("_")
    return col_name

org_df = org_df.toDF(*[clean_column(c) for c in org_df.columns])


# Step 3: Remove Fivetran Metadata Columns
from pyspark.sql.functions import col

org_df = org_df.select([
    col(c) for c in org_df.columns if not c.startswith("_")
])


# Step 4: Basic Data Type Fix
org_df = org_df.withColumn("id", col("id").cast("string"))


# Step 5: Handle Null Values (light)
org_df = org_df.fillna({
    "name": "unknown"
})


# Step 6: Ensure Valid Record
org_df = org_df.filter(col("id").isNotNull())


In [0]:
# Step 7: Write to Silver
org_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.silver.organizations")